<a href="https://colab.research.google.com/github/sureshbudha879-pixel/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

The baseline score identifies content pages that are good candidates for refresh.

The rule prioritizes pages with:

- Older content (not updated for a long time)
- Low click-through rate (CTR)
- Poor average search position
- Declining traffic trend

Each page receives a baseline score, a reason code, and a recommended action.

## Reason Codes

- OLD_CONTENT
- LOW_CTR
- LOW_POSITION
- DECLINING_TRAFFIC

**Note:** This baseline rule is intentionally simple and transparent. It is designed as a decision-support heuristic and will be compared with a machine learning model in later assignments.

In [ ]:
import pandas as pd

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print(df.shape)

df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os
import pandas as pd

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

df["baseline_score"] = 0
df["reason_code"] = ""

# Old content
mask = df["days_since_last_update"] > 365
df.loc[mask, "baseline_score"] += 30
df.loc[mask, "reason_code"] += "OLD_CONTENT "

# Low CTR
mask = df["ctr"] < 0.10
df.loc[mask, "baseline_score"] += 25
df.loc[mask, "reason_code"] += "LOW_CTR "

# Poor search position
mask = df["avg_position"] > 20
df.loc[mask, "baseline_score"] += 20
df.loc[mask, "reason_code"] += "LOW_POSITION "

# Declining traffic
mask = df["trend_direction"] == "down"
df.loc[mask, "baseline_score"] += 25
df.loc[mask, "reason_code"] += "DECLINING_TRAFFIC "

# Action label
df["action"] = "Keep"

df.loc[df["baseline_score"] >= 60, "action"] = "Refresh Immediately"
df.loc[
    (df["baseline_score"] >= 30) &
    (df["baseline_score"] < 60),
    "action"
] = "Review"

queue = df.sort_values(
    "baseline_score",
    ascending=False
)

os.makedirs(
    "/content/flyrank-ml-internship/work/outputs",
    exist_ok=True
)

queue.to_csv(
    "/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv",
    index=False
)

print(queue[
    [
        "content_id",
        "baseline_score",
        "action",
        "reason_code"
    ]
].head(20))

                 content_id  baseline_score               action  \
29384  content_f6fdf87348f6             100  Refresh Immediately   
26242  content_55a5b1c46474              80  Refresh Immediately   
24216  content_1b4ec72dafd4              80  Refresh Immediately   
18440  content_8d56efff1e71              75  Refresh Immediately   
14016  content_4a75dab1c84e              70  Refresh Immediately   
6689   content_e752a4e03dd3              70  Refresh Immediately   
2960   content_312788662f55              70  Refresh Immediately   
21285  content_2e47aaeed913              70  Refresh Immediately   
17988  content_d68a2265c1c0              70  Refresh Immediately   
3022   content_ff1f0c71893f              70  Refresh Immediately   
2992   content_1fb6a9c3aea4              70  Refresh Immediately   
2966   content_bc135bf3bfed              70  Refresh Immediately   
21266  content_67565df5500c              70  Refresh Immediately   
18027  content_bef108055534              70  Ref

In [ ]:
print(df.columns.tolist())

['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = queue.head(20).copy()

top20["confidence_note"] = "Medium"

top20["what_would_make_it_wrong"] = (
    "Traffic decline may be temporary due to seasonality or recent algorithm changes."
)

top20[
    [
        "content_id",
        "baseline_score",
        "action",
        "reason_code",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
]

,content_id,baseline_score,action,reason_code,confidence_note,what_would_make_it_wrong
29384,content_f6fdf87348f6,100,Refresh Immediately,OLD_CONTENT LOW_CTR LOW_POSITION DECLINING_TRA...,Medium,Traffic decline may be temporary due to season...
26242,content_55a5b1c46474,80,Refresh Immediately,OLD_CONTENT LOW_CTR DECLINING_TRAFFIC,Medium,Traffic decline may be temporary due to season...
24216,content_1b4ec72dafd4,80,Refresh Immediately,OLD_CONTENT LOW_CTR DECLINING_TRAFFIC,Medium,Traffic decline may be temporary due to season...
18440,content_8d56efff1e71,75,Refresh Immediately,OLD_CONTENT LOW_CTR LOW_POSITION,Medium,Traffic decline may be temporary due to season...
14016,content_4a75dab1c84e,70,Refresh Immediately,LOW_CTR LOW_POSITION DECLINING_TRAFFIC,Medium,Traffic decline may be temporary due to season...
6689,content_e752a4e03dd3,70,Refresh Immediately,LOW_CTR LOW_POSITION DECLINING_TRAFFIC,Medium,Traffic decline may be temporary due to season...
2960,content_312788662f55,70,Refresh Immediately,LOW_CTR LOW_POSITION DECLINING_TRAFFIC,Medium,Traffic decline may be temporary due to season...
21285,content_2e47aaeed913,70,Refresh Immediately,LOW_CTR LOW_POSITION DECLINING_TRAFFIC,Medium,Traffic decline may be temporary due to season...
17988,content_d68a2265c1c0,70,Refresh Immediately,LOW_CTR LOW_POSITION DECLINING_TRAFFIC,Medium,Traffic decline may be temporary due to season...
3022,content_ff1f0c71893f,70,Refresh Immediately,LOW_CTR LOW_POSITION DECLINING_TRAFFIC,Medium,Traffic decline may be temporary due to season...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages may receive high scores because search traffic changes naturally over time. A temporary seasonal drop or a recent Google update could cause these pages to appear as high priority even if they do not actually require a content refresh.

Manual review is recommended before taking action.

## Leakage Check

The baseline score only uses observed signals:

- days_since_last_update
- ctr
- avg_position
- trend_direction

No future performance, labels, or product-generated flags were used.

The baseline rule is intended only for decision support.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.